---
**Script name:** 04_01_02_station_data_processing.ipynb

**Purpose of script:** This script is to pre-process station data in preparation to correct the corrupted timestamps. 

**Author:** Dr Katie Allan

**Date Created:** 2022-07-10

**Email:** katie.allan@sussex.ac.uk

**License:** 

---

**Notes:** Change 'path_to_folder' to appropriate folder paths. Change 'postgresql://user:@localhost/db_name' to PostgreSQL databse login credentials.

---

## Importing libraries that are required

In [ ]:
#import libraries
import pandas as pd
from pandas.api.types import is_datetime64_ns_dtype
import sqlalchemy as psql
import numpy as np
import os
from shapely.geometry import LineString
import folium
import pickle

## Load Data

In [ ]:
#connect to postgres database
engine = psql.create_engine('postgresql://user:@localhost/db_name')

In [ ]:
#read sql table as pandas.db
bat_data = pd.read_sql_table("bat_data", engine)
test_tag_data = pd.read_sql_table("test_tag_data", engine)

In [13]:
#show the top rows of the dataframe
bat_data.head()

,radio_id,tag_id,node_id,tag_rssi,validated,time,station_id,latitude,longitude,deploy,remove,location,software,bat_name,ring_no,capture_site
0,4,1E4B1907,3289f1,-109,0,2021-09-13 21:08:16+00:00,None,50.59782,-3.602547,2021-09-01 05:00:00+00:00,2021-11-30 09:00:00+00:00,Chudleigh Caves and Woods,V1,Ana,B8063,chudleigh_rock
1,4,1E4B1907,3289f1,-107,0,2021-09-13 21:08:26+00:00,None,50.59782,-3.602547,2021-09-01 05:00:00+00:00,2021-11-30 09:00:00+00:00,Chudleigh Caves and Woods,V1,Ana,B8063,chudleigh_rock
2,4,1E4B1907,3289f1,-110,0,2021-09-13 21:08:35+00:00,None,50.59782,-3.602547,2021-09-01 05:00:00+00:00,2021-11-30 09:00:00+00:00,Chudleigh Caves and Woods,V1,Ana,B8063,chudleigh_rock
3,4,1E4B1907,3289f1,-111,1,2021-09-13 21:08:54+00:00,None,50.59782,-3.602547,2021-09-01 05:00:00+00:00,2021-11-30 09:00:00+00:00,Chudleigh Caves and Woods,V1,Ana,B8063,chudleigh_rock
4,4,1E4B1907,3289f1,-105,0,2021-09-13 21:09:42+00:00,None,50.59782,-3.602547,2021-09-01 05:00:00+00:00,2021-11-30 09:00:00+00:00,Chudleigh Caves and Woods,V1,Ana,B8063,chudleigh_rock


In [14]:
#show the top rows of the dataframe
test_tag_data.head()

,radio_id,tag_id,node_id,tag_rssi,validated,time,station_id,latitude,longitude,deploy,remove,location,software
0,1,55331961,None,-90,0,2022-07-27 19:05:08+00:00,CD74B71612A1,50.864445,-0.083617,2022-05-04 14:27:00+00:00,2022-10-03 16:57:00+00:00,JMS Building,V2
1,2,55331961,None,-97,0,2022-05-18 16:38:25+00:00,CD74B71612A1,50.864445,-0.083617,2022-05-04 14:27:00+00:00,2022-10-03 16:57:00+00:00,JMS Building,V2
2,1,55331961,None,-87,0,2022-07-06 08:22:16+00:00,CD74B71612A1,50.864445,-0.083617,2022-05-04 14:27:00+00:00,2022-10-03 16:57:00+00:00,JMS Building,V2
3,1,55331961,None,-94,0,2022-07-23 09:25:37+00:00,CD74B71612A1,50.864445,-0.083617,2022-05-04 14:27:00+00:00,2022-10-03 16:57:00+00:00,JMS Building,V2
4,1,55331961,None,-96,0,2022-05-19 08:20:48+00:00,CD74B71612A1,50.864445,-0.083617,2022-05-04 14:27:00+00:00,2022-10-03 16:57:00+00:00,JMS Building,V2


In [15]:
#check data structure
print(bat_data.dtypes, test_tag_data.dtypes)

radio_id                      int64
tag_id                       object
node_id                      object
tag_rssi                      int64
validated                     int64
time            datetime64[ns, UTC]
station_id                   object
latitude                    float64
longitude                   float64
deploy          datetime64[ns, UTC]
remove          datetime64[ns, UTC]
location                     object
software                     object
bat_name                     object
ring_no                      object
capture_site                 object
dtype: object radio_id                    int64
tag_id                     object
node_id                    object
tag_rssi                    int64
validated                   int64
time          datetime64[ns, UTC]
station_id                 object
latitude                  float64
longitude                 float64
deploy        datetime64[ns, UTC]
remove        datetime64[ns, UTC]
location                   object
so

In [16]:
#print shape of df
print(bat_data.shape, test_tag_data.shape)

(1654209, 16) (4665587, 13)


## Data processing

In [17]:
#Concatenate the dataframes together
all_data = pd.concat([bat_data, test_tag_data], axis=0)
all_data.head()

,radio_id,tag_id,node_id,tag_rssi,validated,time,station_id,latitude,longitude,deploy,remove,location,software,bat_name,ring_no,capture_site
0,4,1E4B1907,3289f1,-109,0,2021-09-13 21:08:16+00:00,None,50.59782,-3.602547,2021-09-01 05:00:00+00:00,2021-11-30 09:00:00+00:00,Chudleigh Caves and Woods,V1,Ana,B8063,chudleigh_rock
1,4,1E4B1907,3289f1,-107,0,2021-09-13 21:08:26+00:00,None,50.59782,-3.602547,2021-09-01 05:00:00+00:00,2021-11-30 09:00:00+00:00,Chudleigh Caves and Woods,V1,Ana,B8063,chudleigh_rock
2,4,1E4B1907,3289f1,-110,0,2021-09-13 21:08:35+00:00,None,50.59782,-3.602547,2021-09-01 05:00:00+00:00,2021-11-30 09:00:00+00:00,Chudleigh Caves and Woods,V1,Ana,B8063,chudleigh_rock
3,4,1E4B1907,3289f1,-111,1,2021-09-13 21:08:54+00:00,None,50.59782,-3.602547,2021-09-01 05:00:00+00:00,2021-11-30 09:00:00+00:00,Chudleigh Caves and Woods,V1,Ana,B8063,chudleigh_rock
4,4,1E4B1907,3289f1,-105,0,2021-09-13 21:09:42+00:00,None,50.59782,-3.602547,2021-09-01 05:00:00+00:00,2021-11-30 09:00:00+00:00,Chudleigh Caves and Woods,V1,Ana,B8063,chudleigh_rock


In [18]:
#filter by timeframe of second deployment
all_data = all_data.loc[(all_data['time'] >= '2023-09-01') & (all_data['time'] < '2024-01-31')]
all_data.head()

,radio_id,tag_id,node_id,tag_rssi,validated,time,station_id,latitude,longitude,deploy,remove,location,software,bat_name,ring_no,capture_site
659054,1,074B6119,None,-118,0,2023-09-05 20:06:16+00:00,31D096AEC921,50.599312,-3.603671,2023-02-13 17:15:00+00:00,2024-04-15 15:05:29.935897+00:00,Palace Meadow,V3.1,Larry,B8113,chudleigh_rock
659055,1,332A1E78,None,-116,0,2023-09-14 16:05:45+00:00,31D096AEC921,50.599312,-3.603671,2023-02-13 17:15:00+00:00,2024-04-15 15:05:29.935897+00:00,Palace Meadow,V3.1,Usher,B8140,chudleigh_rock
659056,4,2D661E19,3795d0,-110,0,2023-09-20 21:10:02+00:00,None,50.593956,-3.610367,2023-09-20 13:47:00+00:00,2024-04-15 15:05:29.935897+00:00,Chudleigh Bat House,V2,Kay,BA0302,chudleigh_rock
659057,4,2D661E19,3795d0,-110,0,2023-09-20 21:10:11+00:00,None,50.593956,-3.610367,2023-09-20 13:47:00+00:00,2024-04-15 15:05:29.935897+00:00,Chudleigh Bat House,V2,Kay,BA0302,chudleigh_rock
659058,4,2D661E19,3795d0,-109,0,2023-09-20 21:10:39+00:00,None,50.593956,-3.610367,2023-09-20 13:47:00+00:00,2024-04-15 15:05:29.935897+00:00,Chudleigh Bat House,V2,Kay,BA0302,chudleigh_rock


In [19]:
#print shape of df
print(all_data.shape)

(1083566, 16)


In [20]:
#keep only detections from stations
all_data = all_data.loc[all_data['node_id'].isnull()]
all_data.head()

,radio_id,tag_id,node_id,tag_rssi,validated,time,station_id,latitude,longitude,deploy,remove,location,software,bat_name,ring_no,capture_site
659054,1,074B6119,None,-118,0,2023-09-05 20:06:16+00:00,31D096AEC921,50.599312,-3.603671,2023-02-13 17:15:00+00:00,2024-04-15 15:05:29.935897+00:00,Palace Meadow,V3.1,Larry,B8113,chudleigh_rock
659055,1,332A1E78,None,-116,0,2023-09-14 16:05:45+00:00,31D096AEC921,50.599312,-3.603671,2023-02-13 17:15:00+00:00,2024-04-15 15:05:29.935897+00:00,Palace Meadow,V3.1,Usher,B8140,chudleigh_rock
659059,1,4C785261,None,-106,0,2023-09-20 21:11:40+00:00,31D096AEC921,50.599312,-3.603671,2023-02-13 17:15:00+00:00,2024-04-15 15:05:29.935897+00:00,Palace Meadow,V3.1,Joyce,BA0301,chudleigh_rock
659063,1,52616161,None,-109,0,2023-09-20 21:13:29+00:00,31D096AEC921,50.599312,-3.603671,2023-02-13 17:15:00+00:00,2024-04-15 15:05:29.935897+00:00,Palace Meadow,V3.1,Neil,BA0305,chudleigh_rock
659064,1,2D661E19,None,-104,0,2023-09-20 21:13:35+00:00,31D096AEC921,50.599312,-3.603671,2023-02-13 17:15:00+00:00,2024-04-15 15:05:29.935897+00:00,Palace Meadow,V3.1,Kay,BA0302,chudleigh_rock


In [21]:
#print shape of df
print(all_data.shape)

(920214, 16)


In [22]:
#keep one unique record per time point to match node data
all_data = all_data.drop_duplicates(subset=['tag_id', 'time', 'station_id'])
all_data.head()

,radio_id,tag_id,node_id,tag_rssi,validated,time,station_id,latitude,longitude,deploy,remove,location,software,bat_name,ring_no,capture_site
659054,1,074B6119,None,-118,0,2023-09-05 20:06:16+00:00,31D096AEC921,50.599312,-3.603671,2023-02-13 17:15:00+00:00,2024-04-15 15:05:29.935897+00:00,Palace Meadow,V3.1,Larry,B8113,chudleigh_rock
659055,1,332A1E78,None,-116,0,2023-09-14 16:05:45+00:00,31D096AEC921,50.599312,-3.603671,2023-02-13 17:15:00+00:00,2024-04-15 15:05:29.935897+00:00,Palace Meadow,V3.1,Usher,B8140,chudleigh_rock
659059,1,4C785261,None,-106,0,2023-09-20 21:11:40+00:00,31D096AEC921,50.599312,-3.603671,2023-02-13 17:15:00+00:00,2024-04-15 15:05:29.935897+00:00,Palace Meadow,V3.1,Joyce,BA0301,chudleigh_rock
659063,1,52616161,None,-109,0,2023-09-20 21:13:29+00:00,31D096AEC921,50.599312,-3.603671,2023-02-13 17:15:00+00:00,2024-04-15 15:05:29.935897+00:00,Palace Meadow,V3.1,Neil,BA0305,chudleigh_rock
659064,1,2D661E19,None,-104,0,2023-09-20 21:13:35+00:00,31D096AEC921,50.599312,-3.603671,2023-02-13 17:15:00+00:00,2024-04-15 15:05:29.935897+00:00,Palace Meadow,V3.1,Kay,BA0302,chudleigh_rock


In [23]:
#print shape of df
print(all_data.shape)

(564691, 16)


In [24]:
# Convert the 'time' column to datetime and make it timezone-naive
all_data['time'] = pd.to_datetime(all_data['time']).dt.tz_localize(None)
print(all_data.dtypes)

radio_id                      int64
tag_id                       object
node_id                      object
tag_rssi                      int64
validated                     int64
time                 datetime64[ns]
station_id                   object
latitude                    float64
longitude                   float64
deploy          datetime64[ns, UTC]
remove          datetime64[ns, UTC]
location                     object
software                     object
bat_name                     object
ring_no                      object
capture_site                 object
dtype: object


In [25]:
# Group by 'station_id' and create dictionary of dataframes with specified columns
grouped_data = {station_id: group[['time', 'tag_id', 'tag_rssi']] for station_id, group in all_data.groupby('station_id')}

# Rename the columns in each dataframe within the dictionary
renamed_grouped_data = {station_id: df.rename(columns={'tag_id': 'id', 'tag_rssi': 'rssi'}) for station_id, df in grouped_data.items()}

# Displaying the resulting dictionary (for illustration purposes)
for station_id, df in renamed_grouped_data.items():
    print(f"Station ID: {station_id}")
    print(df)
    print()

Station ID: 31D096AEC921
                       time        id  rssi
659054  2023-09-05 20:06:16  074B6119  -118
659055  2023-09-14 16:05:45  332A1E78  -116
659059  2023-09-20 21:11:40  4C785261  -106
659063  2023-09-20 21:13:29  52616161  -109
659064  2023-09-20 21:13:35  2D661E19  -104
...                     ...       ...   ...
3644103 2023-11-21 14:23:20  55331961  -106
3652792 2023-11-22 16:42:17  4B551E4B  -103
3652801 2023-11-22 12:50:14  4B551E4B  -102
3659302 2023-11-01 19:18:25  55331961  -103
3661797 2023-11-23 11:07:35  78780778  -110

[222052 rows x 3 columns]

Station ID: 514F15806A7C
                       time        id  rssi
1185933 2023-10-10 00:51:08  34332A07  -103
1185993 2023-10-10 00:53:41  34332A07  -103
1601715 2023-11-28 17:34:47  33331E55  -110
1636973 2023-12-08 17:49:06  612A3355  -110
1649829 2023-12-19 05:10:13  612A3355  -112
...                     ...       ...   ...
3656828 2023-09-21 15:25:09  55331961  -104
3657192 2023-09-21 03:18:52  55331961  -10

In [26]:
#Check the number of detections in each dataframe
[(key, df.shape) for key, df in renamed_grouped_data.items()]

[('31D096AEC921', (222052, 3)),
 ('514F15806A7C', (3487, 3)),
 ('CD74B71612A1', (253, 3)),
 ('CFF730D7207F', (7938, 3)),
 ('V30B0154B9D5', (195, 3)),
 ('V30B0154BBCF', (405, 3)),
 ('V30B0154C8C9', (63551, 3)),
 ('V30B0154D392', (4520, 3)),
 ('V30B0154D690', (2470, 3)),
 ('V30B0154DB2E', (966, 3)),
 ('V30B0154DDBF', (39365, 3)),
 ('V30B0154EBA8', (219489, 3))]

## Export data

In [ ]:
# Store data (serialize)
with open('output/station_data.pickle', 'wb') as handle:
    pickle.dump(renamed_grouped_data, handle, protocol=pickle.HIGHEST_PROTOCOL)